In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision.transforms.functional as F
import pandas as pd
import numpy as np
import copy
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
from PIL import Image
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. 图像增强与处理函数
# ==========================================
def train_transform_vision(image, label, label_min, label_max):
    augmented_samples = []
    label_normalized = (label - label_min) / (label_max - label_min)
    label_tensor = torch.tensor([label_normalized], dtype=torch.float32)
    image = image.convert("L")
    post_crop_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    for _ in range(5):
        i, j, h, w = transforms.RandomCrop.get_params(image, output_size=(336, 336))
        cropped_image = F.crop(image, i, j, h, w)
        img_rotated = F.rotate(cropped_image, 180)
        img_mirrored = F.hflip(cropped_image)
        augmented_samples.append((post_crop_transform(cropped_image), label_tensor))
        augmented_samples.append((post_crop_transform(img_rotated), label_tensor))
        augmented_samples.append((post_crop_transform(img_mirrored), label_tensor))
    return augmented_samples

def val_test_transform_vision(image, label, label_min, label_max):
    label_normalized = (label - label_min) / (label_max - label_min)
    label_tensor = torch.tensor([label_normalized], dtype=torch.float32)
    image = image.convert("L").resize((336, 336))
    image_tensor = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])(image)
    return [(image_tensor, label_tensor)]

def vision_collate_fn(batch):
    flattened_batch = [item for sublist in batch for item in sublist]
    images, labels = zip(*flattened_batch)
    return torch.stack(images), torch.stack(labels)

# ==========================================
# 2. 纯图像数据集定义
# ==========================================
class VisionDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform, label_min, label_max):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.label_min = label_min
        self.label_max = label_max
        self.img_names = self.df.iloc[:, 0].values
        self.labels = np.log(self.df.iloc[:, -1].values.astype('float32')) # 执行 Log 变换

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, str(self.img_names[idx]))
        image = Image.open(img_path)
        label = self.labels[idx]
        return self.transform(image, label, self.label_min, self.label_max)

# ==========================================
# 3. 纯图像模型 (ResNet18)
# ==========================================
class VisionResNet(nn.Module):
    def __init__(self):
        super(VisionResNet, self).__init__()
        self.net = models.resnet18(pretrained=True)
        self.net.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        num_ftrs = self.net.fc.in_features
        self.net.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(num_ftrs, 1))

    def forward(self, x):
        return self.net(x)

# ==========================================
# 4. 评估核心函数 (还原物理量级)
# ==========================================
def evaluate_set(model, loader, label_min, label_max, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device)).cpu().numpy().flatten()
            labels = labels.numpy().flatten()
            # 还原归一化 -> 还原对数变换
            p_raw = np.exp(outputs * (label_max - label_min) + label_min)
            l_raw = np.exp(labels * (label_max - label_min) + label_min)
            all_preds.extend(p_raw)
            all_labels.extend(l_raw)
    
    r2 = r2_score(all_labels, all_preds)
    rmse = np.sqrt(mean_squared_error(all_labels, all_preds))
    mape = mean_absolute_percentage_error(all_labels, all_preds)
    return r2, rmse, mape

# ==========================================
# 5. 单次实验主逻辑
# ==========================================
def run_single_vision_experiment(seed, label_min, label_max):
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    base_img_dir = './Split_Images' # 确保此路径已物理切分图片
    
    # 构建 DataLoaders
    # 训练集评估时使用简单 transform 以获得稳定指标
    train_ds = VisionDataset('split_train_data.csv', os.path.join(base_img_dir, 'train'), train_transform_vision, label_min, label_max)
    train_eval_ds = VisionDataset('split_train_data.csv', os.path.join(base_img_dir, 'train'), val_test_transform_vision, label_min, label_max)
    val_ds = VisionDataset('split_val_data.csv', os.path.join(base_img_dir, 'val'), val_test_transform_vision, label_min, label_max)
    test_ds = VisionDataset('split_test_data.csv', os.path.join(base_img_dir, 'test'), val_test_transform_vision, label_min, label_max)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=vision_collate_fn)
    train_eval_loader = DataLoader(train_eval_ds, batch_size=16, shuffle=False, collate_fn=vision_collate_fn)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, collate_fn=vision_collate_fn)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=vision_collate_fn)

    model = VisionResNet().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
    criterion = nn.SmoothL1Loss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.3)

    best_val_loss = float('inf')
    best_wts = None

    for epoch in range(15):
        model.train()
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device).unsqueeze(1)
            optimizer.zero_grad()
            loss = criterion(model(imgs), lbls)
            loss.backward()
            optimizer.step()
        
        # 验证
        model.eval()
        v_loss = 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                v_loss += criterion(model(imgs.to(device)), lbls.to(device).unsqueeze(1)).item() * imgs.size(0)
        v_loss /= len(val_ds)
        scheduler.step(v_loss)
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_wts = copy.deepcopy(model.state_dict())

    # 加载最佳模型并全集合评估
    model.load_state_dict(best_wts)
    res_tr = evaluate_set(model, train_eval_loader, label_min, label_max, device)
    res_va = evaluate_set(model, val_loader, label_min, label_max, device)
    res_te = evaluate_set(model, test_loader, label_min, label_max, device)
    
    return res_tr, res_va, res_te

def main():
    # 数据全局标签处理
    df_all = pd.concat([pd.read_csv(f'split_{s}_data.csv') for s in ['train', 'val', 'test']])
    labels_log = np.log(df_all.iloc[:, -1].values.astype('float32'))
    l_min, l_max = labels_log.min(), labels_log.max()

    num_runs = 50
    history = {
        'tr_r2': [], 'tr_rmse': [], 'tr_mape': [],
        'va_r2': [], 'va_rmse': [], 'va_mape': [],
        'te_r2': [], 'te_rmse': [], 'te_mape': []
    }

    print(f"🚀 开始执行 50 次独立的纯图像实验 (Train/Val/Test 全方位评估)...")
    for i in range(num_runs):
        tr, va, te = run_single_vision_experiment(i*42, l_min, l_max)
        
        history['tr_r2'].append(tr[0]); history['tr_rmse'].append(tr[1]); history['tr_mape'].append(tr[2])
        history['va_r2'].append(va[0]); history['va_rmse'].append(va[1]); history['va_mape'].append(va[2])
        history['te_r2'].append(te[0]); history['te_rmse'].append(te[1]); history['te_mape'].append(te[2])
        
        if (i+1) % 5 == 0:
            print(f"   ➤ 已完成 {i+1}/50 次独立实验...")

    # 统计结果
    def get_res(key):
        return f"{np.mean(history[key]):.4f} ± {np.std(history[key]):.4f}"

    final_table = pd.DataFrame([{
        'Model': 'Vision-Only (ResNet18)',
        'Train R2': get_res('tr_r2'), 'Train RMSE': get_res('tr_rmse'), 'Train MAPE': get_res('tr_mape'),
        'Val R2': get_res('va_r2'),   'Val RMSE': get_res('va_rmse'),   'Val MAPE': get_res('va_mape'),
        'Test R2': get_res('te_r2'),  'Test RMSE': get_res('te_rmse'),  'Test MAPE': get_res('te_mape')
    }])

    print("\n" + "="*110)
    print(" "*40 + "Vision-Only 50次实验汇总结果")
    print("="*110)
    print(final_table.to_string(index=False))
    print("="*110)

    final_table.to_csv('Vision_Only_Comprehensive_50Runs.csv', index=False)
    print(f"📊 完整评估数据已保存至: Vision_Only_Comprehensive_50Runs.csv")

if __name__ == '__main__':
    main()

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


🚀 开始执行 50 次独立的纯图像实验 (Train/Val/Test 全方位评估)...


KeyboardInterrupt: 